In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import StandardScaler

# Display all columns
pd.set_option('display.max_columns', None)

In [32]:
# Load the dataset

df = pd.read_csv("../youtube_data.csv")


In [6]:
df.head()

,video_id,title,description,published_date,channel_id,channel_title,tags,category_id,view_count,like_count,comment_count,duration,thumbnail
0,gsJAlLOFBv0,TINY Tech That Actually Works,No description available,2025-05-02T17:37:10Z,UCMiJRAwDNSNzuYeN2uWa0pA,Mrwhosetheboss,"['tiny', 'tech', 'gadgets', 'small', 'miniature']",28,8962092.0,243350.0,515.0,PT57S,https://i.ytimg.com/vi/gsJAlLOFBv0/default.jpg
1,ypicIkaiViM,AI & future of workforce: Andrew Yang on how t...,"Andrew Yang, Forward Party co-chair and former...",2025-06-18T12:39:53Z,UCrp_UI8XtuYfpiqluWLD7Lw,CNBC Television,"['Squawk Box U.S.', 'CNBC', 'business news', '...",25,289626.0,3393.0,1240.0,PT7M50S,https://i.ytimg.com/vi/ypicIkaiViM/default.jpg
2,1Nef8LPO-jo,5 ILLEGAL gadgets that will get you ARRESTED,#shorts #technology \n\nI spend a LOT of time ...,2022-11-01T11:00:06Z,UCMiJRAwDNSNzuYeN2uWa0pA,Mrwhosetheboss,"['shorts', 'tech']",28,81372201.0,4178447.0,6378.0,PT47S,https://i.ytimg.com/vi/1Nef8LPO-jo/default.jpg
3,lCHqmzynO-s,Overrated vs. Underrated Tech,💬 Join my Discord server: https://discord.gg/g...,2024-07-08T18:04:31Z,UCPk2s5c4R_d-EUUNvFFODoA,Gohar Khan,"['thailand', 'surin', 'style', 'travel', 'day'...",27,21255964.0,909386.0,2681.0,PT31S,https://i.ytimg.com/vi/lCHqmzynO-s/default.jpg
4,7uFrtqSwYzM,APPLE Glass Revolutionizes AR Experience Forever!,Discover the revolutionary world of augmented ...,2024-12-22T16:49:00Z,UCxqG_E-68WAE0TWYfIopv6Q,Digifix,"['apple glasses price', 'apple glasses design'...",28,2790436.0,44278.0,1359.0,PT16S,https://i.ytimg.com/vi/7uFrtqSwYzM/default.jpg


In [7]:
df.shape


(600, 13)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        600 non-null    str    
 1   title           600 non-null    str    
 2   description     600 non-null    str    
 3   published_date  600 non-null    str    
 4   channel_id      600 non-null    str    
 5   channel_title   600 non-null    str    
 6   tags            600 non-null    str    
 7   category_id     600 non-null    int64  
 8   view_count      600 non-null    float64
 9   like_count      600 non-null    float64
 10  comment_count   600 non-null    float64
 11  duration        600 non-null    str    
 12  thumbnail       600 non-null    str    
dtypes: float64(3), int64(1), str(9)
memory usage: 61.1 KB


In [9]:
# Display summary statistics of numerical columns

df.describe()

,category_id,view_count,like_count,comment_count
count,600.000000,6.000000e+02,6.000000e+02,600.000000
mean,24.903333,8.080299e+06,2.174646e+05,1970.144781
std,4.863477,2.455377e+07,5.074207e+05,4421.455908
min,1.000000,3.120000e+02,0.000000e+00,0.000000
25%,22.000000,8.150100e+04,1.646000e+03,13.000000
50%,27.000000,9.587180e+05,2.214900e+04,252.500000
75%,28.000000,5.991357e+06,2.174646e+05,1802.500000
max,30.000000,3.437590e+08,4.421091e+06,40241.000000


In [10]:
# Check for missing values in each column

df.isnull().sum()


video_id          0
title             0
description       0
published_date    0
channel_id        0
channel_title     0
tags              0
category_id       0
view_count        0
like_count        0
comment_count     0
duration          0
thumbnail         0
dtype: int64

In [11]:
# Count total missing values

total_missing = df.isnull().sum().sum()

print("Total Missing Values:", total_missing)

Total Missing Values: 0


In [12]:
# Check duplicate rows

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 16


In [13]:
# Remove duplicate rows

df = df.drop_duplicates()

In [14]:
print("Duplicate Rows After Cleaning:", df.duplicated().sum())

Duplicate Rows After Cleaning: 0


In [ ]:

pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'

invalid = df[~df["duration"].astype(str).str.match(pattern, na=False)]

print(invalid[["duration"]])
print("Number of invalid rows:", len(invalid))

    duration
502      P0D
Number of invalid rows: 1


In [39]:
def convert_duration(duration):
    if pd.isna(duration):
        return 0

    duration = str(duration).strip()

    pattern = r'^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$'
    match = re.match(pattern, duration)

    if match is None:
        return 0

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return hours * 3600 + minutes * 60 + seconds

df["duration_seconds"] = df["duration"].astype(str).apply(convert_duration)

In [41]:
df[["duration","duration_seconds"]].head(20)

,duration,duration_seconds
0,PT57S,57
1,PT7M50S,470
2,PT47S,47
3,PT31S,31
4,PT16S,16
5,PT56S,56
6,PT1M,60
7,PT30S,30
8,PT7M17S,437
9,PT1M,60


In [34]:

# Remove unnecessary columns

columns_to_drop = [
    "video_id",
    "title",
    "description",
    "published_date",
    "channel_id",
    "channel_title",
    "tags",
    "thumbnail"
]

df.drop(columns=columns_to_drop, inplace=True)

In [42]:
df.columns


Index(['category_id', 'view_count', 'like_count', 'comment_count', 'duration',
       'duration_seconds'],
      dtype='str')

In [43]:
df.fillna(df.median(numeric_only=True), inplace=True)

,category_id,view_count,like_count,comment_count,duration,duration_seconds
0,28,8962092.0,243350.0,515.0,PT57S,57
1,25,289626.0,3393.0,1240.0,PT7M50S,470
2,28,81372201.0,4178447.0,6378.0,PT47S,47
3,27,21255964.0,909386.0,2681.0,PT31S,31
4,28,2790436.0,44278.0,1359.0,PT16S,16
...,...,...,...,...,...,...
595,24,15631786.0,1692338.0,4662.0,PT1M,60
596,20,616.0,7.0,1.0,PT48S,48
597,27,15534.0,70.0,12.0,PT10S,10
598,22,4223.0,182.0,5.0,PT6S,6


In [44]:
scaler = StandardScaler()

numerical_columns = [
    "view_count",
    "comment_count",
    "duration_seconds",
    "category_id"
]

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

In [18]:
# Save the cleaned dataset

df.to_csv("../processed/cleaned_youtube_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
